# LILA BC

For my project, I need images of wildlife in the Amazon that include animals. I'm using the [LILA BC](https://lila.science/datasets/wcscameratraps) dataset, but since it contains animals from various countries, I filtered the dataset using a JSON file with labels specific to Latin American countries. I also created a script to download the images to my local machine, as I plan to train my model locally. This file represents that work.

## Counting code of countrys

In [14]:
import json
import pandas as pd
# path for JSON file
json_path = 'wcs_camera_traps.json'

# read JSON file
with open(json_path, 'r') as f:
    data = json.load(f)
# count the number of images and select the country codes'
if 'images' in data:
    num_images = len(data['images'])
    df_images = pd.json_normalize(data['images'])
    codes = df_images['country_code'].unique()
print(f"Number of images: {num_images}")
print(codes) 



Number of images: 1369991
['bol' 'ken' 'idn' 'ecu' 'rwa' 'gtm' 'pry' 'tza' 'lao' 'mdg' 'ven' 'uga']


## Country of latin American

- **Bolívia** (bol)
- **Equador** (ecu)
- **Peru** (pry)
- **Venezuela** (ven)


In [14]:
import os

def load_progress(csv_name):
    if(os.path.exists(str(csv_name))):
    # read csv file
        df_names = pd.read_csv(str(csv_name), header=0)
        total = len(df_names)
        print("Foram processadas " + str(total) + " imagens")
        if total > 0:
            # get the last processed image
            last_line = df_names.iloc[-1][1]
            print("Iniciando pelo arquivo " + str(last_line))
        else:
            print("Nenhum arquivo processado")
        return df_names
    
    df_names = pd.DataFrame(columns=['original_name', 'new_name'])
    print("Iniciando do zero as requisições")
    return df_names

def save_progress(df_names, csv_name):
    df_names.to_csv(str(csv_name), mode='a', index=False, header=False)

def filer_dataset(df_names, df_images):
    # filter the dataset to only include images that are not in the CSV file
    if(len(df_names) == 0):
        print("Nenhum arquivo processado")
        return df_images
    df_filtered = df_images[~df_images['file_name'].isin(df_names['original_filename'])]
    return df_filtered

### Creating script for downloand images

In [17]:
import json
import pandas as pd
import requests
import os
import uuid
import time
import csv

# path for JSON file
json_path = 'wcs_camera_traps.json'

# URL for lila dataset
url = 'https://lilawildlife.blob.core.windows.net/lila-wildlife/wcs-unzipped/'

df_names = load_progress('new_names.csv')

# Reading json with python
with open(json_path, 'r') as f:
    data = json.load(f)
if 'images' in data:

    # create dataframe from json image keys
    # data = filer_dataset(df_names, data)
    df_images = pd.json_normalize(data['images'])
    codes = ['bol', 'ecu', 'pry', 'ven']
    data = filer_dataset(df_names, df_images)
    print("Colunas de df_names:", df_names.columns.tolist())
    print("Colunas de data_pandas:", df_images.columns.tolist())

    # filter the dataframe to only include images from the specified latin countries
    total_latinos = df_images[df_images['country_code'].isin(codes)]
    base_path = '../imagens/train/com_animal'

    # create the folder images
    if not os.path.exists(base_path):
        os.makedirs(base_path)

    csv_new_names = []
    # Iterating over the rows of the DataFrame

csv_path = "new_names.csv"

# Garante que o cabeçalho seja escrito apenas uma vez
write_header = not os.path.exists(csv_path)

with open(csv_path, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=["original_filename", "new_filename"], delimiter=';')

    if write_header:
        writer.writeheader()
       
    for i in range(0, 40000):

        #Select name of file and create full url
        original_filename = total_latinos.iloc[i]["file_name"]
        unique_id = str(uuid.uuid4())
        # Nome original
        # Pega a extensão do arquivo original (ex: .jpg, .png)
        _, ext = os.path.splitext(original_filename)
        image_url = url + original_filename
        full_local_path = os.path.join(base_path, "{unique_id}{ext}".format(unique_id=unique_id, ext=ext))
        
        start_time = time.time()  # Marca o início
        response = requests.get(image_url, stream=True)
        end_time = time.time()  # Marca o fim


        if response.status_code == 200:
            # Create subfloders
            os.makedirs(os.path.dirname(full_local_path), exist_ok=True)

            # Write the image to a file
            with open(full_local_path, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
                # Adiciona ao array (para montar o DataFrame depois)
            writer.writerow({
                "original_filename": original_filename,
                "new_filename": f"{unique_id}{ext}"
            })
            print(f"Imagem {original_filename} save.")
            print(f"Tempo da requisição: {end_time - start_time:.2f} segundos")
            print(f"Nome do arquivo:{full_local_path}")
        else:
            print(f"Something wrong with {original_filename}.")
    

Foram processadas 19 imagens
Iniciando pelo arquivo nan


/tmp/ipykernel_3268738/76905848.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  last_line = df_names.iloc[-1][1]


Colunas de df_names: ['original_filename', 'new_filename']
Colunas de data_pandas: ['id', 'wcs_id', 'file_name', 'frame_num', 'seq_id', 'country_code', 'match_level', 'datetime', 'location', 'width', 'height', 'corrupt', 'seq_num_frames', 'status']
Imagem animals/0000/0000.jpg save.
Tempo da requisição: 0.82 segundos
Nome do arquivo:../imagens/train/com_animal/4498d293-19af-4d02-9628-3a4a908003d8.jpg
Imagem animals/0000/0001.jpg save.
Tempo da requisição: 0.75 segundos
Nome do arquivo:../imagens/train/com_animal/2371c555-0670-494a-a5a3-d81b6e4a2ac8.jpg
Imagem animals/0000/0002.jpg save.
Tempo da requisição: 0.74 segundos
Nome do arquivo:../imagens/train/com_animal/9675c2e1-b2ef-4942-b83e-c863f1471733.jpg
Imagem animals/0000/0003.jpg save.
Tempo da requisição: 0.70 segundos
Nome do arquivo:../imagens/train/com_animal/1b9eb63b-2930-46ba-8542-e4056dc91b9f.jpg
Imagem animals/0000/0004.jpg save.
Tempo da requisição: 0.75 segundos
Nome do arquivo:../imagens/train/com_animal/954735c6-38ce-49e

KeyboardInterrupt: 